# Simple Pixel Nearest-Train Audit

This notebook checks whether generated images look like exact/near copies of the Imagenette training set using one simple method: pixel-wise MSE after resizing every image to the same small resolution.

It is intentionally simple and fast. Change the constants in the next cell for a different generated-image folder.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("REPA_ROOT", Path(__file__).resolve().parent if "__file__" in dir() else Path(".")))

# Folder with individual generated PNGs. The notebook ignores grid.png.
GENERATED_DIR = PROJECT_ROOT / "samples" / "repa_100k_model_cfg4_ode250_classes0-9_seed163_notebook_setup"

# Training images exported for REPA
TRAIN_IMAGES_DIR = PROJECT_ROOT / "data" / "imagenette256-train" / "images"

# Small resize for fast pixel comparison. Increase to 128 if you want a slower stricter check.
PIXEL_SIZE = 96

REPORT_DIR = PROJECT_ROOT / "reports" / "nearest_train_pixel_audit" / GENERATED_DIR.name
REPORT_CSV = REPORT_DIR / "nearest_pixel_matches.csv"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
print("Generated folder:", GENERATED_DIR)
print("Train images:", TRAIN_IMAGES_DIR)
print("Pixel comparison size:", PIXEL_SIZE)
print("Report CSV:", REPORT_CSV)

In [ ]:
import math
import re
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from IPython.display import display

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
RESAMPLE = Image.Resampling.BICUBIC if hasattr(Image, "Resampling") else Image.BICUBIC

IMAGENETTE_CLASSES = [
    "tench", "English springer", "cassette player", "chain saw", "church",
    "French horn", "garbage truck", "gas pump", "golf ball", "parachute",
]

def class_name(class_id):
    if class_id is None or pd.isna(class_id):
        return "unknown"
    class_id = int(class_id)
    if 0 <= class_id < len(IMAGENETTE_CLASSES):
        return f"{class_id}: {IMAGENETTE_CLASSES[class_id]}"
    return str(class_id)

In [ ]:
def parse_generated_class(path: Path):
    match = re.search(r"class[-_](\d+)", path.name)
    return int(match.group(1)) if match else None


def parse_train_class(path: Path):
    # REPA export structure: images/00008/img00008000.png
    parent = path.parent.name
    return int(parent) if parent.isdigit() else None


def collect_generated(generated_dir: Path):
    rows = []
    for path in sorted(generated_dir.iterdir()):
        if path.name == "grid.png" or path.suffix.lower() not in IMAGE_EXTS:
            continue
        cid = parse_generated_class(path)
        rows.append({"gen_path": path, "gen_file": path.name, "gen_class_id": cid})
    return pd.DataFrame(rows)


def collect_train(train_dir: Path):
    rows = []
    for path in sorted(train_dir.rglob("*")):
        if path.suffix.lower() not in IMAGE_EXTS:
            continue
        cid = parse_train_class(path)
        rows.append({"train_path": path, "train_file": path.name, "train_class_id": cid})
    return pd.DataFrame(rows)


gen_df = collect_generated(GENERATED_DIR)
train_df = collect_train(TRAIN_IMAGES_DIR)

print("Generated images:", len(gen_df))
print("Train images:", len(train_df))
assert len(gen_df) > 0, f"No generated images found in {GENERATED_DIR}"
assert len(train_df) > 0, f"No train images found in {TRAIN_IMAGES_DIR}"
display(gen_df)
display(train_df.head())

In [ ]:
def load_for_compare(path: Path, size=PIXEL_SIZE):
    image = Image.open(path).convert("RGB")
    image = ImageOps.fit(image, (size, size), method=RESAMPLE, centering=(0.5, 0.5))
    return np.asarray(image, dtype=np.float32) / 255.0


def load_for_display(path: Path):
    return Image.open(path).convert("RGB")


def psnr_from_mse(mse):
    mse = float(mse)
    if mse <= 0:
        return float("inf")
    return 10.0 * math.log10(1.0 / mse)

In [ ]:
# Load all generated images once.
gen_arrays = np.stack([load_for_compare(p) for p in gen_df["gen_path"]], axis=0)

best_mse = np.full(len(gen_df), np.inf, dtype=np.float64)
best_train_index = np.full(len(gen_df), -1, dtype=np.int64)

# Scan the train set once. For each train image, compare it against all generated images.
for train_i, train_path in enumerate(tqdm(train_df["train_path"], desc="Scanning train images")):
    train_arr = load_for_compare(train_path)
    mse = np.mean((gen_arrays - train_arr[None, ...]) ** 2, axis=(1, 2, 3))
    improved = mse < best_mse
    best_mse[improved] = mse[improved]
    best_train_index[improved] = train_i

rows = []
for gen_i, gen_row in gen_df.iterrows():
    train_i = int(best_train_index[gen_i])
    train_row = train_df.iloc[train_i]
    mse = float(best_mse[gen_i])
    rows.append({
        "gen_file": gen_row["gen_file"],
        "gen_path": str(gen_row["gen_path"]),
        "gen_class_id": gen_row["gen_class_id"],
        "gen_class_name": class_name(gen_row["gen_class_id"]),
        "nearest_train_file": train_row["train_file"],
        "nearest_train_path": str(train_row["train_path"]),
        "nearest_train_class_id": train_row["train_class_id"],
        "nearest_train_class_name": class_name(train_row["train_class_id"]),
        "pixel_mse": mse,
        "psnr": psnr_from_mse(mse),
        "same_class": gen_row["gen_class_id"] == train_row["train_class_id"],
    })

result_df = pd.DataFrame(rows).sort_values("pixel_mse").reset_index(drop=True)
result_df.to_csv(REPORT_CSV, index=False)
print("Saved:", REPORT_CSV)
display(result_df)

In [ ]:
n_rows = len(result_df)
fig, axes = plt.subplots(n_rows, 2, figsize=(10, 3.2 * n_rows), squeeze=False)

for idx, (_, row) in enumerate(result_df.iterrows()):
    axes[idx, 0].imshow(load_for_display(Path(row["gen_path"])))
    axes[idx, 0].set_title(
        f"Generated: {row['gen_file']}\n{row['gen_class_name']}",
        fontsize=9,
    )
    axes[idx, 0].axis("off")

    axes[idx, 1].imshow(load_for_display(Path(row["nearest_train_path"])))
    axes[idx, 1].set_title(
        f"Nearest train by pixel MSE: {row['nearest_train_class_name']}\n"
        f"MSE={row['pixel_mse']:.5f}, PSNR={row['psnr']:.1f}, same_class={row['same_class']}",
        fontsize=9,
    )
    axes[idx, 1].axis("off")

fig.suptitle(f"Generated images and nearest train images: {GENERATED_DIR.name}", fontsize=13, y=0.995)
plt.tight_layout(rect=(0, 0, 1, 0.99))

out_path = REPORT_DIR / "nearest_pixel_matches_grid.png"
fig.savefig(out_path, dpi=160, bbox_inches="tight")
print("Saved:", out_path)
plt.show()


In [ ]:
print("How to read this:")
print("- Exact copy: MSE is 0 or extremely close to 0, PSNR is infinite/very high, and the images look identical.")
print("- Not a pixel copy: MSE is clearly nonzero and the nearest train image is visually different.")
print("- This is intentionally only a simple pixel-wise audit. It does not catch high-level semantic similarity unless it also looks pixel-similar after resizing.")
print("CSV:", REPORT_CSV)